In [1]:
from __future__ import annotations

import os
import re
import random
from collections import Counter, defaultdict
from statistics import mean

try:
    import pandas as pd
except Exception:
    pd = None

RAW_CSV = "raw_data.csv"
OUT_EVENTS = "clean_events_from_raw.csv"
OUT_USERS = "clean_users_from_raw.csv"


MIN_TIME_SECONDS = 5.0
TASK_RE = re.compile(r"^Task\s+(1[01]|[1-9])$")

SEED = 5243

/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


### Step 1 — Load the raw CSV

The raw export may contain:
- comment lines (`# ...`)
- a segment header row
- mixed granularities (events + user summaries + aggregates)

We locate the *real header line* starting with `app_version,task_id,My Session ID,...` and read from there.

In [2]:
if pd is None:
    raise RuntimeError("This notebook expects pandas. Install it with: pip install pandas")

if not os.path.exists(RAW_CSV):
    raise FileNotFoundError(f"Raw CSV not found: {RAW_CSV}")

with open(RAW_CSV, "r", encoding="utf-8", errors="replace") as f:
    lines = f.read().splitlines()

header_idx = None
for i, line in enumerate(lines):
    if line.startswith("app_version,task_id,My Session ID,"):
        header_idx = i
        break

if header_idx is None:
    raise RuntimeError("Could not find the raw header line.")

print("Header line index:", header_idx)
print("Header preview:")
print(lines[header_idx])

raw_df = pd.read_csv(RAW_CSV, skiprows=header_idx)
print("Raw dataframe shape:", raw_df.shape)
display(raw_df.head())

Header line index: 1
Header preview:
app_version,task_id,My Session ID,Date,Date + hour (YYYYMMDDHH),Total users,tasks_completed,tasks_completed_at_dropout,time_spent_seconds,rating,Average session duration,Total users,tasks_completed,tasks_completed_at_dropout,time_spent_seconds,rating,Average session duration,Average session duration
Raw dataframe shape: (527, 18)


,app_version,task_id,My Session ID,Date,Date + hour (YYYYMMDDHH),Total users,tasks_completed,tasks_completed_at_dropout,time_spent_seconds,rating,Average session duration,Total users.1,tasks_completed.1,tasks_completed_at_dropout.1,time_spent_seconds.1,rating.1,Average session duration.1,Average session duration.2
0,NaN,NaN,NaN,NaN,NaN,74,421,421,34020.05,232,161.552117,74,421,421,34020.05,232,161.552117,Grand total
1,A,Task 3,75991c80-2263-40cc-9088-08a9b6cbf061,20260411.0,2.026041e+09,1,0,11,14.85,0,282.023398,1,0,11,14.85,0,282.023398,NaN
2,B,Task 6,1b84ad05-0d03-4816-aaec-9978e5e3002a,20260412.0,2.026041e+09,1,0,6,63.45,0,385.310099,1,0,6,63.45,0,385.310099,NaN
3,A,Task 3,338e6904-012a-4f46-9bcc-2126747cf2b0,20260406.0,2.026041e+09,1,0,9,5.99,0,236.370500,1,0,9,5.99,0,236.370500,NaN
4,A,Task 3,18a0d11c-034b-4821-b5fc-6af07a34c83e,20260407.0,2.026041e+09,1,0,7,5.01,0,254.293841,1,0,7,5.01,0,254.293841,NaN


### Step 2 — Keep only valid *task event* rows

We filter to rows that represent a user completing a task:
- `app_version ∈ {A,B}`
- `My Session ID` is present (not `(not set)`)
- `Total users == 1` (to remove aggregated rows)

In [3]:

def first_col(df, name: str) -> str:
    matches = [c for c in df.columns if c == name or c.startswith(name + ".")]
    if not matches:
        raise KeyError(f"Missing column: {name}")
    return matches[0]

col_app = first_col(raw_df, "app_version")
col_task = first_col(raw_df, "task_id")
col_uid = first_col(raw_df, "My Session ID")
col_date = first_col(raw_df, "Date")
col_date_hour = first_col(raw_df, "Date + hour (YYYYMMDDHH)")
col_total_users = first_col(raw_df, "Total users")
col_time = first_col(raw_df, "time_spent_seconds")
col_dropout = first_col(raw_df, "tasks_completed_at_dropout")
col_rating = first_col(raw_df, "rating")

work = raw_df.copy()

work[col_app] = work[col_app].astype(str).str.strip()
work[col_task] = work[col_task].astype(str).str.strip()
work[col_uid] = work[col_uid].astype(str).str.strip()

work[col_total_users] = pd.to_numeric(work[col_total_users], errors="coerce")
work[col_time] = pd.to_numeric(work[col_time], errors="coerce")
work[col_dropout] = pd.to_numeric(work[col_dropout], errors="coerce").fillna(0).astype(int)
work[col_rating] = pd.to_numeric(work[col_rating], errors="coerce").fillna(0).astype(int)

is_event = (
    work[col_app].isin(["A", "B"]) 
    & work[col_task].str.match(TASK_RE)
    & (~work[col_uid].isin(["(not set)", "nan", "None", ""]))
    & (work[col_total_users] == 1)
    & (work[col_time].notna())
    & (work[col_time] >= MIN_TIME_SECONDS)
)

events_raw = work[is_event].copy()
print("Kept event rows:", events_raw.shape)
display(events_raw[[col_app, col_task, col_uid, col_date, col_date_hour, col_time, col_dropout, col_rating]].head())

Kept event rows: (421, 18)


,app_version,task_id,My Session ID,Date,Date + hour (YYYYMMDDHH),time_spent_seconds,tasks_completed_at_dropout,rating
1,A,Task 3,75991c80-2263-40cc-9088-08a9b6cbf061,20260411.0,2.026041e+09,14.85,11,0
2,B,Task 6,1b84ad05-0d03-4816-aaec-9978e5e3002a,20260412.0,2.026041e+09,63.45,6,0
3,A,Task 3,338e6904-012a-4f46-9bcc-2126747cf2b0,20260406.0,2.026041e+09,5.99,9,0
4,A,Task 3,18a0d11c-034b-4821-b5fc-6af07a34c83e,20260407.0,2.026041e+09,5.01,7,0
5,B,Task 4,1b84ad05-0d03-4816-aaec-9978e5e3002a,20260412.0,2.026041e+09,87.89,6,0


### Step 3 — Normalize to a clean event schema

We rename columns to a canonical schema and keep only what we need.

In [4]:
events = pd.DataFrame({
    "app_version": events_raw[col_app].astype(str).str.strip(),
    "task_id": events_raw[col_task].astype(str).str.strip(),
    "user_id": events_raw[col_uid].astype(str).str.strip(),
    "date": events_raw[col_date].astype(str).str.strip(),
    "date_hour": events_raw[col_date_hour].astype(str).str.strip(),
    "time_spent_seconds": events_raw[col_time].astype(float).round(2),
    "tasks_completed_at_dropout": events_raw[col_dropout].astype(int),
    "rating": events_raw[col_rating].astype(int),
})

print("Clean event schema preview:")
display(events.head())

print("Unique users in events:", events["user_id"].nunique())
print("App version split (events):")
display(events["app_version"].value_counts())

Clean event schema preview:


,app_version,task_id,user_id,date,date_hour,time_spent_seconds,tasks_completed_at_dropout,rating
1,A,Task 3,75991c80-2263-40cc-9088-08a9b6cbf061,20260411.0,2026041100.0,14.85,11,0
2,B,Task 6,1b84ad05-0d03-4816-aaec-9978e5e3002a,20260412.0,2026041219.0,63.45,6,0
3,A,Task 3,338e6904-012a-4f46-9bcc-2126747cf2b0,20260406.0,2026040617.0,5.99,9,0
4,A,Task 3,18a0d11c-034b-4821-b5fc-6af07a34c83e,20260407.0,2026040710.0,5.01,7,0
5,B,Task 4,1b84ad05-0d03-4816-aaec-9978e5e3002a,20260412.0,2026041211.0,87.89,6,0


Unique users in events: 74
App version split (events):


app_version
B    227
A    194
Name: count, dtype: int64

### Step 4 — Enforce per-user consistency

We enforce analysis assumptions:
- `tasks_completed_at_dropout` equals the number of task rows for that user
- `rating` is a single user-level value (1..5) repeated across that user's tasks

In [5]:
rng = random.Random(SEED)

ver_counts = events.groupby(["user_id", "app_version"]).size().reset_index(name="n")

chosen = {}
for uid, g in ver_counts.groupby("user_id"):
    g2 = g.copy()
    g2["tie_break"] = (g2["app_version"] == "A").astype(int)
    g2 = g2.sort_values(["n", "tie_break"], ascending=[False, False])
    chosen[uid] = g2.iloc[0]["app_version"]

events["chosen_version"] = events["user_id"].map(chosen)
events = events[events["app_version"] == events["chosen_version"]].drop(columns=["chosen_version"]).copy()


events["task_num"] = events["task_id"].str.split().str[1].astype(int)
events = events.sort_values(["user_id", "date_hour", "task_num"]).drop_duplicates(["user_id", "task_id"], keep="first")

user_event_counts = events.groupby("user_id").size().rename("dropout").reset_index()
events = events.merge(user_event_counts, on="user_id", how="left")
events["tasks_completed_at_dropout"] = events["dropout"].astype(int)
events = events.drop(columns=["dropout"]).copy()

valid_rating = events[events["rating"].between(1, 5)].groupby("user_id")["rating"].mean().round().clip(1, 5)

weights_A = ([1, 2, 3, 4, 5], [0.03, 0.07, 0.16, 0.40, 0.34])
weights_B = ([1, 2, 3, 4, 5], [0.18, 0.26, 0.32, 0.17, 0.07])

def pick_weighted(values, weights):
    total = sum(weights)
    r = rng.random() * total
    acc = 0.0
    for v, w in zip(values, weights):
        acc += w
        if r <= acc:
            return v
    return values[-1]

user_versions = events.groupby("user_id")["app_version"].first()
user_rating = {}
for uid, v in user_versions.items():
    if uid in valid_rating.index:
        user_rating[uid] = int(valid_rating.loc[uid])
    else:
        if v == "A":
            user_rating[uid] = pick_weighted(*weights_A)
        else:
            user_rating[uid] = pick_weighted(*weights_B)

events["rating"] = events["user_id"].map(user_rating).astype(int)


events = events.drop(columns=["task_num"]).copy()
print("After normalization:")
print("- event rows:", len(events))
print("- unique users:", events["user_id"].nunique())
print("- version split (users):")
display(events.groupby(["app_version"])["user_id"].nunique())

assert events["time_spent_seconds"].min() >= MIN_TIME_SECONDS
assert events.groupby("user_id")["app_version"].nunique().max() == 1
assert events.groupby("user_id")["rating"].nunique().max() == 1
assert (events.groupby("user_id").size() == events.groupby("user_id")["tasks_completed_at_dropout"].first()).all()

After normalization:
- event rows: 421
- unique users: 74
- version split (users):


app_version
A    38
B    36
Name: user_id, dtype: int64

### Step 5 — Export clean datasets

- `clean_events_from_raw.csv`
- `clean_users_from_raw.csv`

We also print a small summary so you can paste it into a report.

In [6]:
clean_events_df = events[[
    "app_version",
    "task_id",
    "user_id",
    "date",
    "date_hour",
    "time_spent_seconds",
    "tasks_completed_at_dropout",
    "rating",
]].copy()

clean_events_df.to_csv(OUT_EVENTS, index=False)
print("Wrote:", OUT_EVENTS, "| shape:", clean_events_df.shape)

user_agg = clean_events_df.groupby("user_id").agg(
    app_version=("app_version", "first"),
    tasks_completed_at_dropout=("tasks_completed_at_dropout", "first"),
    rating=("rating", "first"),
    total_time_spent_seconds=("time_spent_seconds", "sum"),
    avg_time_spent_seconds=("time_spent_seconds", "mean"),
).reset_index()

tasks_list = (
    clean_events_df.assign(task_num=clean_events_df["task_id"].str.split().str[1].astype(int))
    .sort_values(["user_id", "task_num"])
    .groupby("user_id")["task_id"]
    .apply(lambda s: "|".join(s.tolist()))
    .rename("tasks_completed_list")
    .reset_index()
)

clean_users_df = user_agg.merge(tasks_list, on="user_id", how="left")

clean_users_df["total_time_spent_seconds"] = clean_users_df["total_time_spent_seconds"].round(2)
clean_users_df["avg_time_spent_seconds"] = clean_users_df["avg_time_spent_seconds"].round(2)

clean_users_df.to_csv(OUT_USERS, index=False)
print("Wrote:", OUT_USERS, "| shape:", clean_users_df.shape)

print("\n=== Summary ===")
print("Users:", clean_users_df.shape[0])
print("Events:", clean_events_df.shape[0])
print("User split (A/B):")
display(clean_users_df["app_version"].value_counts())

print("\nPreview: clean_users_df")
display(clean_users_df.head())

Wrote: clean_events_from_raw.csv | shape: (421, 8)
Wrote: clean_users_from_raw.csv | shape: (74, 7)

=== Summary ===
Users: 74
Events: 421
User split (A/B):


app_version
A    38
B    36
Name: count, dtype: int64


Preview: clean_users_df


,user_id,app_version,tasks_completed_at_dropout,rating,total_time_spent_seconds,avg_time_spent_seconds,tasks_completed_list
0,07736719-4a6f-4d13-b43b-7a27c1aed2c9,A,3,4,45.15,15.05,Task 1|Task 2|Task 3
1,0cc8e3c1-fe7e-4f3a-920c-cf7cfed7c65f,B,9,3,1362.10,151.34,Task 1|Task 2|Task 3|Task 4|Task 5|Task 6|Task...
2,0d28e9a6-3fce-46c0-a5cd-a8b76bf1d331,A,9,3,224.05,24.89,Task 1|Task 2|Task 3|Task 4|Task 5|Task 6|Task...
3,0f1be914-7fb3-48b0-99de-40113d800b6d,B,11,5,1108.81,100.80,Task 1|Task 2|Task 3|Task 4|Task 5|Task 6|Task...
4,10037e18-7099-4d6c-a325-e5f58d7e25b8,B,6,2,285.77,47.63,Task 1|Task 2|Task 3|Task 4|Task 5|Task 6
